# Colab: Gemma v1.1 실행 템플릿
이관본이며 실행하지 않았습니다. 새 실행은 별도 승인 후에만 가능합니다. 원본 평가 문항은 포함하지 않습니다. 기존 216회는 회귀셋이며 새 holdout이 아닙니다.


In [ ]:
from pathlib import Path
import hashlib, json, subprocess, sys, os
ALLOW_EXECUTION = False  # 새 실행 승인과 자원 확인 후에만 변경
if not ALLOW_EXECUTION: raise RuntimeError("Execution disabled")
WORK=Path('/kaggle/working' if 'Colab' == 'Kaggle' else '/content')
ROOT=WORK/'AutoHarness'  # 검토한 checkout 경로
PAYLOAD=WORK/'authorized-inputs'/'execution_payload.json'  # 별도 권한의 입력
assert (ROOT/'tools/gemma4/runner.py').is_file() and PAYLOAD.is_file()


In [ ]:
# 실제 GPU 존재를 확인하며 할당이나 유료 업그레이드를 자동 수행하지 않습니다.
subprocess.run(['nvidia-smi'], check=True)
subprocess.run([sys.executable,'-m','pip','install','huggingface_hub==1.31.0'],check=True)
from huggingface_hub import snapshot_download
MODEL_DIR=Path(snapshot_download(
    repo_id='google/gemma-4-12B-it-qat-q4_0-gguf',
    revision='29d097773436b69ff9feafd636ab4cf873786537',
    allow_patterns=['*.gguf']))
# mmproj와 drafter를 제외하고 실제 모델 파일을 확인합니다.
candidates=[p for p in MODEL_DIR.glob('*.gguf') if not any(x in p.name.lower() for x in ['mmproj','projector','assistant','drafter'])]
print([str(p) for p in candidates])
assert len(candidates)==1, '샤드 또는 여러 후보가 있으면 명시적 파일 선택 후 진행'
MODEL=candidates[0]

In [ ]:
SOURCE=Path('/content/llama.cpp')
COMMIT='11fe02151f79c41d0d4af7da708755d73b9c0da6'
if not SOURCE.exists():
    subprocess.run(['git','clone','https://github.com/ggml-org/llama.cpp.git',str(SOURCE)],check=True)
subprocess.run(['git','checkout','--detach',COMMIT],cwd=SOURCE,check=True)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=SOURCE,text=True).strip()==COMMIT
subprocess.run(['cmake','-S',str(SOURCE),'-B',str(SOURCE/'build'),'-DGGML_CUDA=ON','-DCMAKE_BUILD_TYPE=Release'],check=True)
subprocess.run(['cmake','--build',str(SOURCE/'build'),'--config','Release','-j','2','--target','llama-server'],check=True)
SERVER=SOURCE/'build/bin/llama-server'
VERSION=subprocess.check_output([str(SERVER),'--version'],stderr=subprocess.STDOUT,text=True)
assert '11382' in VERSION and '11fe02151' in VERSION, VERSION
SERVER_SHA256=hashlib.sha256(SERVER.read_bytes()).hexdigest()
print(VERSION)

In [ ]:
# 모델 해시는 로컬 파일에서 계산하여 기록합니다. 대형 파일이므로 시간이 걸릴 수 있습니다.
h=hashlib.sha256()
with MODEL.open('rb') as f:
    for chunk in iter(lambda:f.read(8*1024*1024),b''): h.update(chunk)
OUT=Path('/content/gemma4-run')
OUT.mkdir(exist_ok=True)
# 재연결/런타임 폐기 전에 이 폴더를 Drive 또는 로컬로 복사하세요.
runtime={'repo':'google/gemma-4-12B-it-qat-q4_0-gguf','revision':'29d097773436b69ff9feafd636ab4cf873786537','model_file':MODEL.name,'model_sha256':h.hexdigest(),'llama_commit':COMMIT,'context':4096,'lora':[],'thinking':False,'reasoning_budget':0,'chat_template_kwargs':{'enable_thinking':False},'server_version':VERSION,'server_sha256':SERVER_SHA256}
lock=OUT/'runtime-lock.json'
if lock.exists(): assert json.loads(lock.read_text())==runtime, 'runtime changed; do not resume silently'
else: lock.write_text(json.dumps(runtime,indent=2))
log=(OUT/'server.log').open('a')
server=subprocess.Popen([str(SERVER),'-m',str(MODEL),'-c','4096','-ngl','99','--parallel','1','--host','127.0.0.1','--port','18090','--reasoning-budget','0'],stdout=log,stderr=log)
print('PID',server.pid,'준비 완료 후 다음 셀 실행; OOM이면 기록하고 중단')

## 실행 전 확인
서버 ready, 모델 revision/해시, LoRA 없음, chat/thinking 설정 고정을 확인하세요. 아래 마지막 셀만 생성 요청을 보냅니다. 기본 코드는 개발셋만 실행합니다. 기존 실행 승인은 새 실행으로 이어지지 않습니다. 모든 프롬프트/판정 기준은 실행 전에 고정하며 개발 결과로 바꾸지 않습니다. 예약된 요청이 중단되면 자동 재전송하지 않습니다.

In [ ]:
import importlib.util
spec=importlib.util.spec_from_file_location('colab_run',ROOT/'tools/gemma4/runner.py')
runner=importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)
INCLUDE_EVALUATION=False  # 추가 입력 실행은 별도 승인 필요
CONFIRM_RUNTIME=False   # 바로 위 확인사항을 확인한 뒤 True
state=runner.run(PAYLOAD,OUT,include_evaluation=INCLUDE_EVALUATION,confirm_runtime=CONFIRM_RUNTIME)
print('완료',sum(r['status']=='completed' for r in state['records'].values()))

In [ ]:
# 런타임이 사라지기 전에 결과와 checkpoint를 저장합니다.
import shutil
archive=shutil.make_archive('/content/gemma4-run-checkpoint','zip',OUT)
print(archive)  # 승인된 비공개 보관 경로로 수동 보존
# 새 런타임에서는 같은 payload와 모델/설정으로 서버를 복원하고,
# 이 ZIP을 OUT에 먼저 복원한 뒤 runner.run을 호출합니다.
# completed만 건너뛰며 reserved/실패는 수동 확인 전 자동 재시도하지 않습니다.